# DQN Gridworld: environment design and reward shaping

This project originated from an **MMA 606 course assignment**. The training notebook was **adapted from the course-provided starter workflow**. The experiment extends a crystal-collection gridworld with a larger wrapping map, swamp terrain, dash actions, a deeper Q-network, and a proximity reward. It also explores supervised action prediction from the replay buffer using an RBF-kernel SVM.

This public notebook preserves the submitted implementation and its active settings. It is an exploratory experiment, not evidence of reliable convergence or an optimal policy. The companion report describes several different historical trials; the saved notebook run has a final mean return of **−33**, while a separate report screenshot shows **1013** for another trial. These are distinct records.

Outputs and execution counts were cleared for publication. The original [training log](../assets/notebook/training_log.txt), [51-step trajectory](../assets/notebook/trajectory.txt), and [SVM classification report](../assets/notebook/svm_classification_report.txt) are retained separately. The notebook's original SVM confusion matrix is byte-identical to the [report image](../assets/report/svm_confusion_matrix.png), so only one copy is stored. No DQN training was rerun during repository preparation.

## Environment setup

Install the repository's `requirements.txt` in a Python environment, then launch Jupyter and run the cells in order from a fresh kernel. The implementation uses PyTorch on its default device, NumPy, scikit-learn, Matplotlib, and Seaborn. This is a custom environment; it does not use the Gymnasium API. Original dependency versions and random seeds were not recorded, so a new run need not match the saved results.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from copy import deepcopy

## Training configuration

The submitted settings are retained: discount 0.99, replay capacity 100, batch size 32, target-network synchronization every 20 completed episodes, Adam learning rate 0.1, up to 200 training episodes, and a 50-step training limit. Exploration starts at 1.0 and decays exponentially toward 0.05 with denominator 200.

In [ ]:
# Discount factor
GAMMA = 0.99

# Batch size
BATCH_SIZE = 32
# Capacity of the replay buffer
BUFFER_CAPACITY = 100
# Update target net every ... episodes
UPDATE_TARGET_EVERY = 20

# Initial value of epsilon
EPSILON_START = 1.0
# Parameter to decrease epsilon
DECREASE_EPSILON = 200
# Minimum value of epislon
EPSILON_MIN = 0.05

# Number of training episodes
N_EPISODES = 200

# Learning rate
LEARNING_RATE = 0.1

# Max rollout length
ROLLOUT_LENGTH = 50

## Toroidal gridworld

The active map is 6×6. Both movement and the agent's 3×3 observation wrap across its boundaries. Each observed cell is one-hot encoded over six tokens (`A`, `C`, `R`, `F`, `-`, `S`), giving **54 input features**. The agent observes a local window rather than the full map.

The code indexes the map as `map[x][y]`, with `x` as the outer list. Actions 0/1 decrement/increment `x` (named left/right); 2/3 decrement/increment `y` (up/down). Actions 4–7 dash three cells in the corresponding direction. The two crystals, `C` and `R`, have the same reward and collection rule.

Reward and transition details are preserved exactly:

- A normal move starts at −1. Landing on swamp adds −2, a crystal adds +5, and collecting the last crystal adds another +1000. Fire adds −1000 and terminates the episode.
- A dash starts at −2 and checks each traversed cell. Each swamp encountered adds −1. Any fire immediately stops the dash and ends the episode. Crystals are collected only when the third, final cell is reached, not while passing through them.
- After updating the map, the beacon adds `2 * max(0, 3 - d)`, where `d` is the nearest **remaining** crystal's toroidal Manhattan distance. No crystals means no bonus. This bonus is also applied on fire termination. It is a per-step proximity bonus, not a potential-difference shaping formula.
- Entered terrain is overwritten by the agent token and becomes empty after departure. In particular, a swamp landing cell can disappear; intermediate dash cells are not cleared. This is part of the submitted environment, not a publication-time change.

In [ ]:
#Defining the basics of the environment
class Environment:
	def __init__(self, map):
		#String matrix representation of the map, A: Agent, C: Crystal, R: Crystal 2, F: Fire, -: Empty
		self.map = map
		self.initialMap = deepcopy(map)
		self.totalCrystals = 0
		self.playerPos = []


		self.values = ["A", "C", "R", "F", "-", "S"]
		self.observationSize = 3*3*len(self.values)
		self.defaultToken = "-"
		self.playerToken = "A"
		self.numActions = 8
		self.reset()
	def beacon_reward(self):
		min_dist = float('inf')
		for x in range(len(self.map)):
			for y in range(len(self.map[0])):
				if self.map[x][y] in ["C", "R"]:
					dx = min(abs(self.playerPos[0]-x), len(self.map)-abs(self.playerPos[0]-x))
					dy = min(abs(self.playerPos[1]-y), len(self.map[0])-abs(self.playerPos[1]-y))
					dist = dx + dy
					min_dist = min(min_dist, dist)
		if min_dist < float('inf'):
			return max(0, 3 - min_dist)*2
		return 0
	def reset(self): #Reset the environment to the initial map values
		self.map = deepcopy(self.initialMap)
		self.totalCrystals = 0
		self.playerPos = []
		for x in range(0, len(self.map)):
			for y in range(0, len(self.map[0])):#assumes non-jagged
				if self.map[x][y]=="A":
					self.playerPos = [x,y]
				elif self.map[x][y]=="C":
					self.totalCrystals+=1
				elif self.map[x][y]=="R":
					self.totalCrystals+=1
		return self.observation()

	#Converts an environment into an observation as a 3x3 window centred on the agent
	def observation(self): #Get one hot representation of observation
		x = self.playerPos[0]
		y = self.playerPos[1]
		observation = []
		#grab 3x3
		for xi in range(x-1, x+2):
			for yi in range(y-1, y+2):

				xi_wrapped = xi % len(self.map)
				yi_wrapped = yi % len(self.map[0])
				oneHot = [0]*len(self.values)
				oneHot[self.values.index(self.map[xi_wrapped][yi_wrapped])] = 1
				observation.append(oneHot)
		return np.array(observation).flatten()

	def observationStr(self): #Get string representation of observation
		x = self.playerPos[0]
		y = self.playerPos[1]
		observation = ""

		for xi in range(x-1, x+2):
			for yi in range(y-1, y+2):
				xi_wrapped = xi % len(self.map)
				yi_wrapped = yi % len(self.map[0])
				observation += self.map[xi_wrapped][yi_wrapped]
		return observation

	#next_state, reward, done = env.step(action)
	#Take one step (specified by action) in the environment
	def step(self, action):
		reward = -1 #default reward value
		terminal = False #Whether or not this is a terminal state

		oldPosition = list(self.playerPos)#Grab last position

		#Handle actions
		if len(self.playerPos)>0: #If we still have a player
			if action==0: #Move left
				self.playerPos[0] = (self.playerPos[0] - 1) % len(self.map)
			elif action==1: #Move right
				self.playerPos[0] = (self.playerPos[0] + 1) % len(self.map)
			elif action==2: #Move up
				self.playerPos[1] = (self.playerPos[1] - 1) % len(self.map[0])
			elif action==3: #Move down
				self.playerPos[1] = (self.playerPos[1] + 1) % len(self.map[0])
			elif action >= 4 and action <= 7:
					reward = -2
					direction = {4: (-1, 0), 5: (1, 0), 6: (0, -1), 7: (0, 1)}
					dx, dy = direction[action]
					for step_i in range(3):
							self.playerPos[0] = (self.playerPos[0] + dx) % len(self.map)
							self.playerPos[1] = (self.playerPos[1] + dy) % len(self.map[0])
							cell = self.map[self.playerPos[0]][self.playerPos[1]]
							if cell == "F":
									reward += -1000
									terminal = True
									break
							if step_i < 2:
									if cell == "S":
											reward += -1
							else:
									if cell == "S":
											reward += -1
									elif cell == "C" or cell == "R":
											self.totalCrystals -= 1
											reward += 5
											if self.totalCrystals == 0:
													reward += 1000
													terminal = True
					self.map[oldPosition[0]][oldPosition[1]] = self.defaultToken
					self.map[self.playerPos[0]][self.playerPos[1]] = self.playerToken
					reward += self.beacon_reward()
					return self.observation(), reward, terminal
		#Logic to update map
		if self.map[self.playerPos[0]][self.playerPos[1]]=="C" or self.map[self.playerPos[0]][self.playerPos[1]]=="R": #Collect crystals
			self.totalCrystals -=1
			reward += 5 #Reward for crystals
			if self.totalCrystals ==0:
				reward +=1000 #Reward for collecting all crystals
				terminal = True
		elif self.map[self.playerPos[0]][self.playerPos[1]]=="F":
			reward += -1000 #Reward for moving into fire
			terminal = True
		elif self.map[self.playerPos[0]][self.playerPos[1]]=="S":
			reward += -2

		#Update map
		self.map[oldPosition[0]][oldPosition[1]] = self.defaultToken
		self.map[self.playerPos[0]][self.playerPos[1]] = self.playerToken
		reward += self.beacon_reward()
		return self.observation(), reward, terminal

### Active map

The start is `(3, 2)`, crystals are at `(0, 1)` and `(0, 5)`, using the code's `(x, y)` convention. The active map has swamp at the top-left array entry and **fire at the bottom-right**. The report describes changing the bottom-right fire to swamp during its iteration history; this notebook does not retain that map variant.

In [ ]:
map = [
    ["S", "R", "F", "S", "-", "C"],
    ["S", "F", "-", "-", "-", "F"],
    ["-", "F", "-", "F", "-", "-"],
    ["-", "-", "A", "-", "S", "-"],
    ["-", "S", "S", "F", "-", "-"],
    ["-", "S", "S", "-", "-", "F"]
    ]

In [ ]:
env = Environment(map)

## Experience replay

The circular buffer retains at most 100 transitions. Training samples 32 transitions uniformly without replacement once enough data are available. These same stored states and action labels are later used by the SVM.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        """Saves a transition."""
        if len(self.memory) < self.capacity:
            self.memory.append((state, action, reward, next_state, done))
        else:
          self.memory[self.position] = (state, action, reward, next_state, done)
          self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, k=batch_size)


    def __len__(self):
        return len(self.memory)

# create instance of replay buffer
replay_buffer = ReplayBuffer(BUFFER_CAPACITY)

## Q-network, target network, and optimizer

The network is **54 → 256 → 128 → 8**, with ReLU after both hidden layers, for 48,008 parameters. The online network predicts the Q-value of each action; a separate target network supplies the bootstrapped target. Mean squared error and Adam match the submitted experiment.

The two networks are initialized independently. The original implementation does not synchronize them at initialization; the first copy occurs after 20 completed episodes. This limitation is retained so that the archived experiment is not silently changed.

In [ ]:
class Net(nn.Module):
    """
    Basic neural net.
    """
    def __init__(self, obs_size, hidden_size, n_actions):
        super(Net, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Linear(hidden_size//2, n_actions)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# create network and target network
hidden_size = 256
obs_size = env.observationSize# Observation Size
n_actions = env.numActions #Number of Actions

q_net = Net(obs_size, hidden_size, n_actions)
target_net = Net(obs_size, hidden_size, n_actions)

# objective and optimizer
objective = nn.MSELoss()
optimizer = optim.Adam(params=q_net.parameters(), lr=LEARNING_RATE)

## Greedy evaluation and DQN updates

Evaluation resets a copy of the same fixed map and uses epsilon zero. The final 20 rollouts repeat the same deterministic start and policy; they are not 20 unseen test maps. The evaluation and verbose-rollout loops use `rolloutStep <= ROLLOUT_LENGTH`, allowing **51 steps**, while training uses a 50-step limit. This historical difference is preserved.

In [ ]:
#
#  Some useful functions
#

def get_q(states):
    """
    Compute Q function for a list of states
    """
    with torch.no_grad():
        states_v = torch.FloatTensor([states])
        output = q_net.forward(states_v).data.numpy()  # shape (1, len(states), n_actions)
    return output[0, :, :]  # shape (len(states), n_actions)

def eval_dqn(n_sim=1):
    """
    Monte Carlo evaluation of DQN agent.

    Repeat n_sim times:
        * Run the DQN policy until the environment reaches a terminal state (= one episode)
        * Compute the sum of rewards in this episode
        * Store the sum of rewards in the episode_rewards array.
    """
    env_copy = deepcopy(env)
    episode_rewards = np.zeros(n_sim)

    for ii in range(n_sim):
        state = env_copy.reset()
        done = False
        rolloutStep = 0
        while not done and rolloutStep<=ROLLOUT_LENGTH:
            action = choose_action(state, 0.0)
            next_state, reward, done = env_copy.step(action)
            episode_rewards[ii] += reward
            state = next_state
            rolloutStep+=1
    return episode_rewards

In [ ]:
def choose_action(state, epsilon):
    """
    Return action according to an epsilon-greedy exploration policy
    """
    if np.random.rand() < epsilon:
        return random.randint(0,env.numActions-1)
    else:
        with torch.no_grad():
            state_v = torch.FloatTensor(state).unsqueeze(0)
            q_values = q_net.forward(state_v)
            action = torch.argmax(q_values).item()
        return action


def update(state, action, reward, next_state, done):
    # add data to replay buffer
    replay_buffer.push(state, action, reward, next_state, done)

    if len(replay_buffer) < BATCH_SIZE:
        return np.inf

    # get batch
    transitions = replay_buffer.sample(BATCH_SIZE)

    states, actions, rewards, next_states, dones = zip(*transitions)

    states_v = torch.FloatTensor(states)
    actions_v = torch.LongTensor(actions).unsqueeze(-1)
    rewards_v = torch.FloatTensor(rewards).unsqueeze(-1)
    next_states_v = torch.FloatTensor(next_states)
    dones_v = torch.BoolTensor(dones).unsqueeze(-1)

    # Compute Q-values for current states and chosen actions
    q_values = q_net(states_v).gather(1, actions_v)

    # Compute target Q-values
    with torch.no_grad():
        next_state_q_values = target_net(next_states_v).max(1)[0].unsqueeze(-1)
        target_q_values = rewards_v + GAMMA * next_state_q_values * (~dones_v)

    # Compute loss
    loss = objective(q_values, target_q_values)

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.data.numpy()

## Train and inspect the saved-run context

The training loop periodically evaluates the greedy policy and stops early if its mean reward reaches 1000. The submitted run did not reach that threshold: its largest printed checkpoint return is 99 and its final mean is −33.

The code increments the completed-episode counter before evaluating and printing `ep + 1`. Consequently, the checkpoint labeled 5 follows four completed episodes, and the checkpoint labeled 200 follows 199. Training still allows 200 completed episodes, followed by the final evaluation. These labels are retained in both the code and original log.

Running the next cell starts a new stochastic training experiment. Historical output is available in [training_log.txt](../assets/notebook/training_log.txt).

In [ ]:
#
# Train
#

EVAL_EVERY = 5
REWARD_THRESHOLD = 1000

def train():
    state = env.reset()
    epsilon = EPSILON_START
    ep = 0
    rolloutStep = 0
    while ep < N_EPISODES:
        action = choose_action(state, epsilon)

        # take action and update replay buffer and networks
        next_state, reward, done = env.step(action)
        loss = update(state, action, reward, next_state, done)

        # update state
        state = next_state
        rolloutStep+=1

        # end episode if done
        if done or rolloutStep>=ROLLOUT_LENGTH:
            state = env.reset()
            rolloutStep = 0
            ep   += 1

            if ( (ep+1)% EVAL_EVERY == 0):
                rewards = eval_dqn()
                print("episode =", ep+1, ", reward = ", np.mean(rewards))
                if np.mean(rewards) >= REWARD_THRESHOLD:
                    break



            # update target network
            if ep % UPDATE_TARGET_EVERY == 0:
                target_net.load_state_dict(q_net.state_dict())
            # decrease epsilon
            epsilon = EPSILON_MIN + (EPSILON_START - EPSILON_MIN) * \
                            np.exp(-1. * ep / DECREASE_EPSILON )


# Run the training loop
train()

# Evaluate the final policy
rewards = eval_dqn(20)
print("")
print("mean reward after training = ", np.mean(rewards))

## Inspect a greedy trajectory

This cell prints the action, wrapped local observation, and reward after each step. The saved 51-step trajectory collects `R` on step 2, leaves `C` uncollected, and then repeats a loop until the evaluation limit. Its rewards sum to −33. Replaying the recorded actions through the unchanged environment verifies those transitions; it does not reproduce the learned policy or training process.

In [ ]:
env_copy = deepcopy(env)

state = env_copy.reset()
done = False
rolloutStep = 0
while not done and rolloutStep<=ROLLOUT_LENGTH:
    action = choose_action(state, 0.0)
    next_state, reward, done = env_copy.step(action)
    state = next_state
    rolloutStep+=1
    print ("Step : "+str(rolloutStep))
    print ("Action: "+str(action))
    stringObservation = env_copy.observationStr()
    print ("State: ")
    print (stringObservation[0:3]+"\n"+stringObservation[3:6]+"\n"+stringObservation[6:9])
    print ("Reward: "+str(reward))

## SVM action prediction on the replay buffer

An RBF-kernel SVM learns action labels from all states retained in the replay buffer. It is fitted and scored on **the same data**: there is no held-out split, cross-validation, or separate episode evaluation. The labels include exploratory training actions, so this is not a clean imitation dataset of only the final greedy policy.

The saved classification report contains 100 transitions and **64% training-data accuracy**, with macro F1 0.41 and weighted F1 0.59. This score does not measure navigation success or performance on unseen states. Buffer contents and action frequencies depend on the unseeded training run.

In [ ]:
from sklearn.svm import SVC

In [ ]:
states = []
actions = []
for s, a, r, ns, d in replay_buffer.memory:
    states.append(s)
    actions.append(a)
states = np.array(states)
actions = np.array(actions)

In [ ]:
svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(states, actions)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

### Report all eight actions

Both the classification report and confusion matrix explicitly use `labels=list(range(8))`. This publication-time display fix keeps labels aligned and permits reporting when a new run's buffer lacks one or more action classes. It does not alter SVM fitting, predictions, or the original saved metrics.

In [ ]:
action_names = ['Left','Right','Up','Down','Dash-L','Dash-R','Dash-U','Dash-D']
pred_actions = svm_rbf.predict(states)
print(classification_report(actions, pred_actions, labels=list(range(8)), target_names=action_names, zero_division=0))

In [ ]:
cm = confusion_matrix(actions, pred_actions, labels=list(range(8)))
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=action_names, yticklabels=action_names)
plt.xlabel('Predicted Action')
plt.ylabel('Actual Action')
plt.title('SVM Action Prediction - Confusion Matrix')
plt.tight_layout()
plt.show()

## Interpretation and reproducibility limits

The report's iteration history motivates reward shaping as a hypothesis. The active notebook preserves a run that still loops and leaves a crystal uncollected. The report and notebook should therefore be read as exploratory records across differing configurations and runs, not a controlled ablation or a guarantee of successful learning.

Map geometry, network architecture, and reward scales changed across reported trials. Shaped return itself changes when the reward changes. There are no recorded seeds, repeated-seed statistics, saved model weights, or held-out-map results. The per-step beacon can reward proximity without collection, and the local observation does not encode the entire mutable environment. These limitations constrain claims about convergence, optimal routes, and generalization.

See [README](../README.md) and [experiment notes](../docs/experiment_notes.md) for the source record and historical results. A future controlled study should measure crystal completion and path length separately from shaped return; that would be a new experiment rather than a retroactive correction of these results.